# Final Modeling Notebook

This notebook is the execution layer for the final phase. It trains the two propensity models, evaluates them with a simple temporal holdout, parses the CRM rule file, and exports score and decision outputs.

Run `eda.ipynb` first so the shared snapshot artifacts are written to `artifacts/eda_training_artifacts.pkl`.

In [1]:
from pathlib import Path
import json
from typing import Any, cast

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from IPython.display import display

PROJECT_ROOT = Path('.')
EDA_ARTIFACT_PATH = PROJECT_ROOT / 'artifacts' / 'eda_training_artifacts.pkl'
RULES_PATH = PROJECT_ROOT / 'rules' / 'decision_rules.txt'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'final'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not EDA_ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {
            EDA_ARTIFACT_PATH}. Run eda.ipynb first to materialize the shared training artifacts."
    )

In [2]:
artifacts = pd.read_pickle(EDA_ARTIFACT_PATH)
snapshot_features = artifacts['snapshot_features'].copy()
snapshot_labels = artifacts['snapshot_labels'].copy()

snapshots = snapshot_features.merge(
    snapshot_labels,
    on=['idSSO', 'reference_date'],
    how='inner',
)
snapshots['reference_date'] = pd.to_datetime(snapshots['reference_date'])
snapshots['reference_month'] = snapshots['reference_date'].dt.strftime('%Y-%m')
snapshots['points_gap_proxy'] = (
    snapshots['reward_threshold_points'] - snapshots['totalPoints'].fillna(0)
).clip(lower=0)

churn_frame = snapshots[
    snapshots['days_since_last_activity'].between(30, 59, inclusive='both')
    & snapshots['churn_30_to_60'].notna()
].copy()
churn_frame['churn_30_to_60'] = churn_frame['churn_30_to_60'].astype(int)

redemption_frame = snapshots[
    (snapshots['is_points_user'] == 1)
    & snapshots['days_since_last_activity'].le(90)
].copy()
redemption_frame['will_redeem_30d'] = redemption_frame['will_redeem_30d'].astype(
    int)

shared_features = artifacts['feature_columns'] + \
    ['points_gap_proxy', 'reference_month']
categorical_features = ['child_age_bucket', 'platform', 'reference_month']

print('Churn rows:', len(churn_frame))
print('Redemption rows:', len(redemption_frame))
print('Reference dates:', snapshots['reference_date'].min(
).date(), '->', snapshots['reference_date'].max().date())

Churn rows: 5787
Redemption rows: 32183
Reference dates: 2024-04-01 -> 2025-08-01


## Modeling Approach

Both propensities use the same simple stack: shared snapshot features, a temporal holdout on the most recent four reference months, and a class-weighted logistic regression with basic preprocessing. The goal here is reproducibility and an easy handoff, not an over-engineered training framework.

In [3]:
def make_deciles(probabilities: pd.Series) -> pd.Series:
    ranked = probabilities.rank(method='first')
    deciles = pd.qcut(ranked, 10, labels=range(1, 11), duplicates='drop')
    return pd.Series(deciles, index=probabilities.index, dtype='Int64')


def split_by_time(frame: pd.DataFrame, holdout_months: int = 4) -> tuple[pd.DataFrame, pd.DataFrame]:
    reference_dates = sorted(frame['reference_date'].dropna().unique())
    if len(reference_dates) <= holdout_months:
        raise ValueError(
            'Not enough reference dates for the requested holdout split.')
    valid_dates = set(reference_dates[-holdout_months:])
    train_df = cast(
        pd.DataFrame, frame.loc[~frame['reference_date'].isin(valid_dates)].copy())
    valid_df = cast(
        pd.DataFrame, frame.loc[frame['reference_date'].isin(valid_dates)].copy())
    return cast(tuple[pd.DataFrame, pd.DataFrame], (train_df, valid_df))


def build_pipeline(feature_columns: list[str], categorical_columns: list[str]) -> Pipeline:
    usable_categoricals = [
        col for col in categorical_columns if col in feature_columns]
    numeric_columns = [
        col for col in feature_columns if col not in usable_categoricals]
    preprocess = ColumnTransformer(
        transformers=[
            (
                'numeric',
                Pipeline(steps=[('imputer', SimpleImputer(
                    strategy='median')), ('scaler', StandardScaler())]),
                numeric_columns,
            ),
            (
                'categorical',
                Pipeline(steps=[('imputer', SimpleImputer(
                    strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]),
                usable_categoricals,
            ),
        ]
    )
    return Pipeline(
        steps=[
            ('preprocess', preprocess),
            ('model', LogisticRegression(class_weight='balanced',
             max_iter=1000, random_state=42, solver='liblinear')),
        ]
    )


def calibration_table(frame: pd.DataFrame, probability_column: str, label_column: str) -> pd.DataFrame:
    scored = frame.copy()
    scored['score_decile'] = make_deciles(scored[probability_column])
    return (
        scored.groupby('score_decile', dropna=False)
        .agg(rows=(label_column, 'size'), avg_prediction=(probability_column, 'mean'), actual_rate=(label_column, 'mean'))
        .reset_index()
        .sort_values('score_decile')
    )


def evaluate_predictions(frame: pd.DataFrame, probability_column: str, label_column: str) -> pd.Series:
    y_true = frame[label_column].astype(int)
    y_prob = frame[probability_column].clip(1e-6, 1 - 1e-6)
    top_n = max(1, int(np.ceil(len(frame) * 0.10)))
    top_rate = float(frame.nlargest(top_n, probability_column)
                     [label_column].mean())
    return pd.Series({
        'rows': int(len(frame)),
        'positive_rate': float(y_true.mean()),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'average_precision': float(average_precision_score(y_true, y_prob)),
        'brier_score': float(brier_score_loss(y_true, y_prob)),
        'log_loss': float(log_loss(y_true, y_prob)),
        'top_decile_precision': top_rate,
    })


def fit_propensity_model(frame: pd.DataFrame, *, label_column: str, score_column: str, feature_columns: list[str], categorical_columns: list[str]) -> dict[str, Any]:
    usable_features = [
        col for col in feature_columns if col in frame.columns and not cast(pd.Series, frame[col]).dropna().empty]
    model_frame = frame.copy()
    for column in usable_features:
        if column in categorical_columns:
            model_frame[column] = model_frame[column].astype(
                'string').fillna('missing').astype(str)
        else:
            model_frame[column] = pd.to_numeric(
                model_frame[column], errors='coerce').astype(float)

    train_df, valid_df = split_by_time(model_frame)
    validation_model = build_pipeline(usable_features, categorical_columns)
    validation_model.fit(train_df[usable_features],
                         train_df[label_column].astype(int))

    valid_scored = valid_df.copy()
    valid_scored[score_column] = validation_model.predict_proba(
        valid_df[usable_features])[:, 1]

    final_model = build_pipeline(usable_features, categorical_columns)
    final_model.fit(model_frame[usable_features],
                    model_frame[label_column].astype(int))

    full_scored = model_frame.copy()
    full_scored[score_column] = final_model.predict_proba(
        model_frame[usable_features])[:, 1]

    feature_names = final_model.named_steps['preprocess'].get_feature_names_out(
    )
    coefficients = pd.DataFrame(
        {'feature': feature_names, 'coefficient': final_model.named_steps['model'].coef_[0]})
    coefficients['abs_coefficient'] = coefficients['coefficient'].abs()
    coefficients = coefficients.sort_values(
        'abs_coefficient', ascending=False).head(15)

    return {
        'validation_model': validation_model,
        'final_model': final_model,
        'features': usable_features,
        'validation_scores': valid_scored,
        'full_scores': full_scored,
        'validation_metrics': evaluate_predictions(valid_scored, score_column, label_column),
        'calibration': calibration_table(valid_scored, score_column, label_column),
        'top_coefficients': coefficients,
    }

In [4]:
churn_results: dict[str, Any] = fit_propensity_model(
    churn_frame,
    label_column='churn_30_to_60',
    score_column='churn_30_to_60_prob',
    feature_columns=shared_features,
    categorical_columns=categorical_features,
)

redemption_results: dict[str, Any] = fit_propensity_model(
    redemption_frame,
    label_column='will_redeem_30d',
    score_column='redeem_30d_prob',
    feature_columns=shared_features,
    categorical_columns=categorical_features,
)

validation_metrics = pd.DataFrame([
    {'model': 'churn', **churn_results['validation_metrics'].to_dict()},
    {'model': 'redemption', **
        redemption_results['validation_metrics'].to_dict()},
]).round(4)

validation_metrics

,model,rows,positive_rate,roc_auc,average_precision,brier_score,log_loss,top_decile_precision
0,churn,1251.0,0.6563,0.7981,0.8763,0.2188,0.6288,0.9524
1,redemption,9674.0,0.0185,0.6375,0.0320,0.2489,0.7205,0.0496


In [5]:
def load_rules(path: Path) -> list[dict[str, str]]:
    rules: list[dict[str, str]] = []
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        parts = {}
        for part in line.split('|'):
            key, value = part.split('=', 1)
            parts[key] = value
        rules.append(parts)
    return rules


def apply_rules(scored_frame: pd.DataFrame, audience: str, rules: list[dict[str, str]]) -> pd.DataFrame:
    segment_field = 'churn_type' if audience == 'churn' else 'activation_segment'
    action_field = 'recommended_campaign' if audience == 'churn' else 'recommended_action'
    decisions = pd.DataFrame(index=scored_frame.index)
    decisions['matched_rule'] = pd.NA
    decisions[segment_field] = pd.NA
    decisions[action_field] = pd.NA
    decisions['recommended_channel'] = pd.NA
    decisions['priority'] = pd.NA
    remaining = pd.Series(True, index=scored_frame.index)
    for rule in [item for item in rules if item['audience'] == audience]:
        if rule['condition'] == 'True':
            rule_mask = remaining.copy()
        else:
            rule_mask = scored_frame.eval(
                rule['condition'], engine='python').fillna(False).astype(bool)
            rule_mask &= remaining
        decisions.loc[rule_mask, 'matched_rule'] = rule['rule_name']
        decisions.loc[rule_mask, segment_field] = rule['segment']
        decisions.loc[rule_mask, action_field] = rule['action']
        decisions.loc[rule_mask,
                      'recommended_channel'] = rule['recommended_channel']
        decisions.loc[rule_mask, 'priority'] = rule['priority']
        remaining &= ~rule_mask
    base_columns = ['idSSO', 'reference_date']
    if audience == 'churn':
        base_columns += ['churn_30_to_60_prob', 'risk_decile']
    else:
        base_columns += ['redeem_30d_prob', 'points_gap_proxy']
    return pd.concat([scored_frame[base_columns], decisions], axis=1)


rules = load_rules(RULES_PATH)

churn_scores = churn_results['full_scores'][[
    'idSSO', 'reference_date', 'churn_30_to_60_prob']].copy()
churn_scores['risk_decile'] = make_deciles(churn_scores['churn_30_to_60_prob'])
churn_engine_input = churn_results['full_scores'].copy()
churn_engine_input['risk_decile'] = make_deciles(
    churn_engine_input['churn_30_to_60_prob'])
churn_engine = apply_rules(churn_engine_input, 'churn', rules)

redemption_scores = redemption_results['full_scores'][[
    'idSSO', 'reference_date', 'redeem_30d_prob', 'points_gap_proxy']].copy()
redemption_engine = apply_rules(
    redemption_results['full_scores'], 'redemption', rules)

latest_reference_date = snapshots['reference_date'].max()
current_outputs = {
    'churn_scores_current': churn_scores[churn_scores['reference_date'] == latest_reference_date],
    'redemption_scores_current': redemption_scores[redemption_scores['reference_date'] == latest_reference_date],
    'churn_engine_current': churn_engine[churn_engine['reference_date'] == latest_reference_date],
    'redemption_engine_current': redemption_engine[redemption_engine['reference_date'] == latest_reference_date],
}

for name, frame in {
    'churn_scores_all': churn_scores,
    'redemption_scores_all': redemption_scores,
    'churn_engine_all': churn_engine,
    'redemption_engine_all': redemption_engine,
    **current_outputs,
}.items():
    frame.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)

validation_metrics.to_csv(OUTPUT_DIR / 'validation_metrics.csv', index=False)
churn_results['calibration'].to_csv(
    OUTPUT_DIR / 'churn_calibration.csv', index=False)
redemption_results['calibration'].to_csv(
    OUTPUT_DIR / 'redemption_calibration.csv', index=False)
churn_results['top_coefficients'].to_csv(
    OUTPUT_DIR / 'churn_top_coefficients.csv', index=False)
redemption_results['top_coefficients'].to_csv(
    OUTPUT_DIR / 'redemption_top_coefficients.csv', index=False)

with (OUTPUT_DIR / 'run_summary.json').open('w') as handle:
    json.dump({
        'latest_reference_date': str(latest_reference_date.date()),
        'rules_path': str(RULES_PATH),
        'eda_artifact_path': str(EDA_ARTIFACT_PATH),
        'churn_rows': int(len(churn_scores)),
        'redemption_rows': int(len(redemption_scores)),
    }, handle, indent=2)

print('Saved outputs to', OUTPUT_DIR)
validation_metrics

Saved outputs to artifacts/final


,model,rows,positive_rate,roc_auc,average_precision,brier_score,log_loss,top_decile_precision
0,churn,1251.0,0.6563,0.7981,0.8763,0.2188,0.6288,0.9524
1,redemption,9674.0,0.0185,0.6375,0.0320,0.2489,0.7205,0.0496


In [6]:
print('Churn calibration (validation holdout)')
display(churn_results['calibration'])

print('Redemption calibration (validation holdout)')
display(redemption_results['calibration'])

print('Churn engine sample')
display(churn_engine.head(10))

print('Redemption engine sample')
display(redemption_engine.head(10))

Churn calibration (validation holdout)


,score_decile,rows,avg_prediction,actual_rate
0,1,126,0.131421,0.285714
1,2,125,0.202870,0.336000
2,3,125,0.253262,0.408000
3,4,125,0.308315,0.536000
4,5,125,0.366578,0.696000
5,6,125,0.434623,0.736000
6,7,125,0.523822,0.800000
7,8,125,0.654059,0.944000
8,9,125,0.758254,0.872000
9,10,125,0.863484,0.952000


Redemption calibration (validation holdout)


,score_decile,rows,avg_prediction,actual_rate
0,1,968,0.126968,0.006198
1,2,967,0.214426,0.008273
2,3,967,0.278181,0.015512
3,4,968,0.341111,0.015496
4,5,967,0.402021,0.017580
5,6,967,0.465326,0.014478
6,7,968,0.529597,0.019628
7,8,967,0.606977,0.013444
8,9,967,0.697343,0.024819
9,10,968,0.855728,0.049587


Churn engine sample


,idSSO,reference_date,churn_30_to_60_prob,risk_decile,matched_rule,churn_type,recommended_campaign,recommended_channel,priority
25,095278ec491f259c3430bed05bf39a907bde85d396c49b...,2024-04-01,0.747856,8,default_churn,mixed_or_uncertain,insufficient_channel_eligibility,none,low
40,e71c5a177b0c96d6be1448eb21105dd09ea6d9f18a5198...,2024-04-01,0.973697,10,preventable_push,preventable_churn,preventable_churn_core,push,high
47,9449088472699f3887acee3b5ce296472ea1a89331ad19...,2024-04-01,0.949918,10,preventable_push,preventable_churn,preventable_churn_core,push,high
64,a387de25c621a0bdbe823995be6402b9322c2b37524a58...,2024-04-01,0.268523,3,default_churn,mixed_or_uncertain,insufficient_channel_eligibility,none,low
83,0eb7a27640f6d28409bd9a4fb43700491893a36da5b0c3...,2024-04-01,0.100212,1,default_churn,mixed_or_uncertain,insufficient_channel_eligibility,none,low
101,56b1bd933953fab9e44690cb07362770653114ce6da362...,2024-04-01,0.715339,8,default_churn,mixed_or_uncertain,insufficient_channel_eligibility,none,low
115,16d56e836858281fc7434dda7d354722909b1bc12a5dd6...,2024-04-01,0.617837,6,default_churn,mixed_or_uncertain,insufficient_channel_eligibility,none,low
120,0a43f40eaaab5f42e03dedf03a8f792d5c4f7db30da248...,2024-04-01,0.187492,2,default_churn,mixed_or_uncertain,insufficient_channel_eligibility,none,low
142,0a9c6bf154af70279bca1ce241198c715319b660486699...,2024-04-01,0.188389,2,default_churn,mixed_or_uncertain,insufficient_channel_eligibility,none,low
144,a1aa15552c24fd215e968fd8993873eace14b8c6b74f52...,2024-04-01,0.625175,6,default_churn,mixed_or_uncertain,insufficient_channel_eligibility,none,low


Redemption engine sample


,idSSO,reference_date,redeem_30d_prob,points_gap_proxy,matched_rule,activation_segment,recommended_action,recommended_channel,priority
6,13138414e028eff18f0b705cebe5cb5c28bcd645358efd...,2024-04-01,0.015769,0.0,default_redemption,ready_to_redeem,points_balance_reminder,email,low
8,e29ef81dda850778b56930bbea87228a84fb58abde2a13...,2024-04-01,0.049967,0.0,not_eligible,not_campaign_eligible,no_action,none,low
19,6260b7930804e6d735f5a9bedf6dec42725544c7fa6443...,2024-04-01,0.048517,0.0,default_redemption,ready_to_redeem,points_balance_reminder,email,low
26,1f33ef18f2c977721c5cca94de37daf11cb87df640eff8...,2024-04-01,0.042329,0.0,default_redemption,ready_to_redeem,points_balance_reminder,email,low
38,b0ca094a858a0d5c7ac0de352cfca6e56055682de08f77...,2024-04-01,0.043397,0.0,default_redemption,ready_to_redeem,points_balance_reminder,email,low
41,6c59f309ac3c06f5574117216e15a26e28b65dff5fa019...,2024-04-01,0.051423,0.0,default_redemption,ready_to_redeem,points_balance_reminder,email,low
45,d51e88d093cc7fc5432678bfc4f691edce0a9b725d5cb2...,2024-04-01,0.039572,5260.0,default_redemption,ready_to_redeem,points_balance_reminder,email,low
50,f4f568c51ba65986c148291329525c8c96ba64969282d0...,2024-04-01,0.022638,0.0,default_redemption,ready_to_redeem,points_balance_reminder,email,low
55,11e6a4d50ccd32cb2ccfe0aa7b62987d6408290e9f7c70...,2024-04-01,0.033147,0.0,default_redemption,ready_to_redeem,points_balance_reminder,email,low
57,1a68b65cb7a3f7403005892ac96754706e9bceee8dd309...,2024-04-01,0.006914,4020.0,default_redemption,ready_to_redeem,points_balance_reminder,email,low
